# Session 3: File Handling & Logging

**Course:** Python for Data Engineering  
**Phase 1:** Foundations

**What we'll cover:**
- Reading and writing files (CSV, TXT, JSON)
- File modes (read, write, append)
- Introduction to logging
- Lab: Build a file-based ETL pipeline with logging

**Data files:** We'll use files in the `data/` directory — same data you've been working with in Sessions 1 & 2 (sales, employees, transactions, logs).

---

## 1. Reading & Writing Text Files

The basic pattern for file I/O in Python:

```python
with open("filename", "mode") as f:
    # do something with f
```

The `with` statement automatically closes the file when you're done — always use it.

### File modes

| Mode | What it does |
|------|--------------|
| `"r"` | Read (default). File must exist. |
| `"w"` | Write. Creates file or **overwrites** existing. |
| `"a"` | Append. Creates file or adds to the end. |
| `"r+"` | Read and write. File must exist. |

In [ ]:
# Reading a text file — the pipeline log from Session 1

with open("data/pipeline.log", "r") as f:
    content = f.read()

print(content)

In [ ]:
# Reading line by line — more memory efficient for large files

with open("data/pipeline.log", "r") as f:
    for line in f:
        line = line.strip()  # remove trailing newline
        print(line)

In [ ]:
# Read all lines into a list

with open("data/pipeline.log", "r") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")
print(f"First line: {lines[0].strip()}")
print(f"Last line: {lines[-1].strip()}")

In [ ]:
# Parse the log file — same parsing logic from Session 1, now from a file

parsed_logs = []

with open("data/pipeline.log", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(" | ")
        parsed_logs.append({
            "timestamp": parts[0],
            "level": parts[1],
            "event": parts[2],
            "user_id": int(parts[3].split("=")[1]),
            "ip": parts[4].split("=")[1],
        })

print(f"Parsed {len(parsed_logs)} log entries")
for log in parsed_logs:
    print(f"  [{log['level']}] {log['event']} — user {log['user_id']}")

In [ ]:
# Writing a text file

# Filter errors from parsed logs and write to a separate file
errors = [log for log in parsed_logs if log["level"] == "ERROR"]

with open("data/errors_only.log", "w") as f:
    for err in errors:
        f.write(f"{err['timestamp']} | {err['event']} | user_id={err['user_id']}\n")

print(f"Wrote {len(errors)} error entries to data/errors_only.log")

# Verify by reading it back
with open("data/errors_only.log", "r") as f:
    print(f.read())

In [ ]:
# Appending to a file — adding more entries without overwriting

new_entry = "2024-01-15 11:00:00 | ERROR | timeout | user_id=1004 | ip=10.0.0.8"

with open("data/errors_only.log", "a") as f:
    f.write(f"{new_entry}\n")

# Check
with open("data/errors_only.log", "r") as f:
    print(f.read())

**Try it:**

1. Read `data/pipeline.log` and count how many lines have each log level (INFO, ERROR, WARNING)
2. Write only the WARNING entries to `data/warnings.log`
3. Read back `data/warnings.log` and print it

In [ ]:
# Your code here


---

## 2. Working with CSV Files

Python has a built-in `csv` module. You don't need pandas for simple CSV work.

Two main tools:
- `csv.reader` — reads rows as lists
- `csv.DictReader` — reads rows as dictionaries (usually what you want)

In [ ]:
import csv

# csv.reader — rows come as lists
with open("data/sales.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)  # first row is the header
    print(f"Columns: {header}")
    for row in reader:
        print(row)

In [ ]:
# csv.DictReader — rows come as dictionaries (much nicer)

with open("data/sales.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

In [ ]:
# Read, clean, and store — same cleaning logic from Session 2!

clean_sales = []
invalid = []

with open("data/sales.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        quantity = int(row["quantity"])
        if quantity <= 0:
            invalid.append(row)
            continue
        
        price = float(row["unit_price"].replace("$", ""))
        clean_sales.append({
            "date": row["date"],
            "product": row["product"].strip().title(),
            "quantity": quantity,
            "unit_price": price,
            "total": round(quantity * price, 2),
            "region": row["region"].strip().lower(),
        })

print(f"Clean: {len(clean_sales)}, Invalid: {len(invalid)}")
for sale in clean_sales:
    print(f"  {sale['product']:12s} | qty={sale['quantity']} | ${sale['total']:>10,.2f} | {sale['region']}")

In [ ]:
# Writing CSV files — save clean data to a new file

output_fields = ["date", "product", "quantity", "unit_price", "total", "region"]

with open("data/sales_clean.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=output_fields)
    writer.writeheader()
    writer.writerows(clean_sales)

print(f"Wrote {len(clean_sales)} records to data/sales_clean.csv")

# Verify
with open("data/sales_clean.csv", "r") as f:
    print(f.read())

**Try it:**

1. Read `data/employees.csv` using `csv.DictReader`
2. Clean each record (same rules as Session 1 Lab 2):
   - Strip + title case the name
   - Convert salary (remove $ then float)
   - Convert age to int
   - Skip rows where department is "unknown"
3. Write the clean records to `data/employees_clean.csv`
4. Read back and print to verify

In [ ]:
import csv

# Your code here


---

## 3. Working with JSON Files

JSON is everywhere in data engineering — API responses, config files, NoSQL data. Python's built-in `json` module handles it.

| Function | What it does |
|----------|--------------|
| `json.load(file)` | Read JSON from a file |
| `json.loads(string)` | Parse JSON from a string |
| `json.dump(data, file)` | Write JSON to a file |
| `json.dumps(data)` | Convert to JSON string |

In [ ]:
import json

# Read JSON — the transaction data from Session 1
with open("data/transactions.json", "r") as f:
    transactions = json.load(f)

print(f"Loaded {len(transactions)} transactions")
print(f"Type: {type(transactions)}")

for txn in transactions:
    print(f"  {txn['txn_id']}: {txn['customer']} bought {txn['quantity']}x {txn['product']}")

In [ ]:
# Process and enrich the data — add total_amount like in Session 1 Lab 3

for txn in transactions:
    txn["total_amount"] = round(txn["quantity"] * txn["unit_price"], 2)

# Write enriched data back to JSON
with open("data/transactions_enriched.json", "w") as f:
    json.dump(transactions, f, indent=2)

print("Wrote enriched transactions to data/transactions_enriched.json")

# Verify
with open("data/transactions_enriched.json", "r") as f:
    print(f.read())

In [ ]:
# json.dumps — convert to string (useful for printing, sending over network)

sample = transactions[0]
print(json.dumps(sample, indent=2))

# Without indent — compact (how APIs send it)
print(json.dumps(sample))

In [ ]:
# json.loads — parse a JSON string (common when working with API responses)

api_response = '{"status": "ok", "count": 42, "data": [{"id": 1}, {"id": 2}]}'

parsed = json.loads(api_response)
print(f"Status: {parsed['status']}")
print(f"Count: {parsed['count']}")
print(f"Data: {parsed['data']}")

In [ ]:
# Writing a config file as JSON — very common in DE

pipeline_config = {
    "pipeline_name": "sales_etl",
    "source": {
        "type": "csv",
        "path": "data/sales.csv"
    },
    "target": {
        "type": "csv",
        "path": "data/sales_clean.csv"
    },
    "settings": {
        "skip_invalid": True,
        "log_level": "INFO"
    }
}

with open("data/pipeline_config.json", "w") as f:
    json.dump(pipeline_config, f, indent=2)

print("Saved pipeline config")

# Read it back — how a pipeline would load its config
with open("data/pipeline_config.json", "r") as f:
    config = json.load(f)

print(f"Pipeline: {config['pipeline_name']}")
print(f"Source: {config['source']['path']}")
print(f"Target: {config['target']['path']}")

**Try it:**

1. Read `data/transactions.json`
2. Group transactions by customer (use a dict — remember `group_by` from Session 2?)
3. Create a summary dict like: `{"Alice Johnson": {"orders": 2, "total_spent": 1159.49}, ...}`
4. Write this summary to `data/customer_summary.json`

In [ ]:
import json

# Your code here


---

## 4. Introduction to Logging

In real pipelines, you don't use `print()` — you use Python's `logging` module. It gives you:
- Log levels (DEBUG, INFO, WARNING, ERROR, CRITICAL)
- Timestamps automatically
- Output to file and console
- Easy to turn on/off different levels

### Log Levels

| Level | When to use |
|-------|-----------|
| `DEBUG` | Detailed diagnostic info (not shown by default) |
| `INFO` | General progress updates — "loaded 100 rows" |
| `WARNING` | Something unexpected but not breaking — "3 rows had nulls" |
| `ERROR` | Something failed but pipeline can continue — "failed to parse row 5" |
| `CRITICAL` | Pipeline must stop — "database connection lost" |

In [ ]:
import logging

# Basic setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger("pipeline")

# Using different levels
logger.debug("This won't show — level is INFO")  # below INFO, hidden
logger.info("Pipeline started")
logger.warning("3 rows had missing values")
logger.error("Failed to connect to backup DB")
logger.critical("Main database is down!")

In [ ]:
# Logging to a file — what you'd do in production

# Create a new logger with file output
file_logger = logging.getLogger("file_pipeline")
file_logger.setLevel(logging.INFO)

# Remove any existing handlers to avoid duplicates in notebooks
file_logger.handlers.clear()

# Add file handler
file_handler = logging.FileHandler("data/pipeline_run.log", mode="w")
file_handler.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
))
file_logger.addHandler(file_handler)

# Also log to console
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))
file_logger.addHandler(console_handler)

# Simulate a pipeline with logging
file_logger.info("Pipeline started")
file_logger.info("Reading data/sales.csv")
file_logger.info("Loaded 8 records")
file_logger.warning("2 records had invalid quantity — skipped")
file_logger.info("Cleaned 6 records")
file_logger.info("Wrote output to data/sales_clean.csv")
file_logger.info("Pipeline completed")

# Check the log file
print("\n--- Log file contents ---")
with open("data/pipeline_run.log", "r") as f:
    print(f.read())

In [ ]:
# Putting it together: a mini pipeline with proper logging

import csv
import json
import logging

# Setup logger
log = logging.getLogger("sales_pipeline")
log.setLevel(logging.INFO)
log.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))
log.addHandler(handler)

# EXTRACT
log.info("Starting extract phase")
raw_sales = []
with open("data/sales.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw_sales.append(row)
log.info(f"Extracted {len(raw_sales)} records from sales.csv")

# TRANSFORM
log.info("Starting transform phase")
clean = []
skipped = 0
for row in raw_sales:
    qty = int(row["quantity"])
    if qty <= 0:
        log.warning(f"Skipping invalid row: {row['product'].strip()} has quantity={qty}")
        skipped += 1
        continue
    price = float(row["unit_price"].replace("$", ""))
    clean.append({
        "date": row["date"],
        "product": row["product"].strip().title(),
        "quantity": qty,
        "unit_price": price,
        "total": round(qty * price, 2),
        "region": row["region"].strip().lower(),
    })
log.info(f"Transformed {len(clean)} records, skipped {skipped}")

# LOAD
log.info("Starting load phase")
fields = ["date", "product", "quantity", "unit_price", "total", "region"]
with open("data/sales_clean.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(clean)
log.info(f"Loaded {len(clean)} records to data/sales_clean.csv")

total_revenue = sum(r["total"] for r in clean)
log.info(f"Pipeline complete — total revenue: ${total_revenue:,.2f}")

**Try it:**

Add logging to any pipeline. Replace all `print()` with proper logging calls:
- `logger.info()` for progress updates
- `logger.warning()` for skipped/bad records
- `logger.error()` for failures

Try setting the level to `DEBUG` vs `WARNING` and see what changes.

In [ ]:
# Your code here


---

## Lab Exercises

---

### Lab 1: CSV → Clean → CSV Pipeline

Read `data/employees.csv`, clean it, and write to `data/employees_report.csv`.

**Steps:**
1. Read with `csv.DictReader`
2. Clean: strip+title name, convert salary (remove $), convert age to int
3. Skip rows where department is `"unknown"`
4. Add fields: `annual_bonus` (salary × 10%), `senior` (True if age >= 35)
5. Write to `data/employees_report.csv`
6. Add logging for each step

In [ ]:
import csv
import logging

# Your code here


---

### Lab 2: JSON → Aggregate → JSON

Read `data/transactions.json`, aggregate by product, write summary to JSON.

**Steps:**
1. Read transactions from JSON
2. For each product, calculate: total quantity sold, total revenue, number of orders
3. Sort products by total revenue (highest first)
4. Write the summary to `data/product_summary.json`

Expected output format:
```json
[
  {"product": "Laptop", "total_qty": 2, "total_revenue": 1999.98, "orders": 2},
  ...
]
```

In [ ]:
import json

# Your code here


---

### Lab 3: Full ETL Pipeline with Logging

Build a complete pipeline that:

1. **Reads config** from `data/pipeline_config.json`
2. **Extracts** data from the source file specified in config
3. **Transforms** the data (clean, filter, enrich)
4. **Loads** to the target file specified in config
5. **Logs** every step to both console and `data/etl_run.log`
6. **Writes a summary** to `data/etl_summary.json` with:
   - total records read
   - records cleaned
   - records skipped
   - total revenue
   - run timestamp

This is your first real pipeline that reads config, processes files, and produces output + logs!

In [ ]:
import csv
import json
import logging
from datetime import datetime

# Your code here


---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| Text files | `open()` with `with` statement. Modes: `r`, `w`, `a` |
| CSV | `csv.DictReader` to read, `csv.DictWriter` to write |
| JSON | `json.load`/`json.dump` for files, `json.loads`/`json.dumps` for strings |
| Logging | Use `logging` module, not `print()`. Levels: DEBUG → INFO → WARNING → ERROR → CRITICAL |

**Key patterns:**
- Always use `with open(...)` — it handles closing the file
- `csv.DictReader` gives you dicts, which is easier to work with than lists
- JSON ↔ Python dict is a direct mapping — very natural
- Log to file in production, console during development
- A real pipeline: read config → extract → transform → load → log everything

**Next session:** Exception Handling & Debugging — making your pipelines fault-tolerant.